# Descriptor Computation

Compute descriptors from one or more `vasprun.xml` files.


## Imports


In [ ]:
import numpy as np
from dscribe.descriptors import SOAP
from tqdm import tqdm

from src.desc_comp_utils import fixed_mask, force_components
from src.workflow_config import discover_structure_files, ensure_directory
from src.workflow_io import build_filemap, build_provenance_table, clear_directory_outputs, load_structure_sets, save_descriptor_run, unique_species


## Input Data


### Define input roots


In [ ]:
# Define the input roots for the system you want to process.
# Use one or more directories, or pass explicit structure file paths.
input_roots = ["./data"]  # replace with your system-specific folders
vasp_files = discover_structure_files(input_roots, patterns=["vasprun*.xml", "XDATCAR*", "*.xyz", "*.extxyz", "*.traj"], recursive=True)
print(f"Found {len(vasp_files)} structure files across {len(input_roots)} input roots.")

### Load structures


In [ ]:
# Read all structures from the VASP files with ASE
structures = load_structure_sets(vasp_files)
print(f"Total number of structures read: {sum(len(s) for s in structures)}")

# Get unique atomic species in the structures
species = unique_species(structures)
print(f"Species found in structures: {species}")

### Optional: inspect a frame


In [ ]:
print(structures[0][0])  # Print the first structure for verification

### Optional: override species selection


In [ ]:
# Convert the different species into a single one (e.g., "C") if needed
modify_species = False  # Set to False to keep original species
new_species = list(species)

if modify_species:
    print("Modifying all species to a single type.")
    target_species = "C"
    for struct_list in structures:
        for atoms in struct_list:
            for atom in atoms:
                if atom.symbol != target_species:
                    atom.symbol = target_species

    new_species = list(set(atom.symbol for struct_list in structures for atoms in struct_list for atom in atoms))
    print(f"Species after conversion: {new_species}")


## Compute SOAP descriptors


In [ ]:
# Define SOAP parameters
soap_params = {
    "species": new_species,
    "periodic": True,
    "r_cut": 6.0,
    "n_max": 4,
    "l_max": 4,
    "sigma": 1.0,
    "compression": {"mode": "mu2"}  # Use "mu2" compression to reduce descriptor size
}

if "compression" in soap_params:
    print(f"Using compression mode: {soap_params['compression']['mode']}")

# Initialize SOAP descriptor
soap = SOAP(**soap_params)
print("SOAP descriptor initialized with parameters:", soap_params)

# Optional: append force components to SOAP vectors
include_forces = False  # set True to append force components
force_components_idx = (0, 1)  # choose components, e.g. (0,1,2) for Fx,Fy,Fz

# --- compute + provenance
total_structs = sum(len(s) for s in structures)
desc_blocks = []
force_blocks = []

pbar = tqdm(total=total_structs, desc="Computing SOAP with provenance")
for struct_list in structures:
    for atoms in struct_list:
        D = soap.create(atoms, n_jobs=1)  # shape: (n_atoms, dim)
        desc_blocks.append(D)
        if include_forces:
            F = force_components(atoms, components=force_components_idx)
            force_blocks.append(F)
        pbar.update(1)
pbar.close()

all_soap_descriptors = np.vstack(desc_blocks)
if include_forces:
    all_forces = np.vstack(force_blocks)
    if np.isnan(all_forces).any():
        print("Forces not found. The forces addition has been skipped and only SOAP descriptors are kept.")
        include_forces = False
        all_descriptors = all_soap_descriptors
    else:
        all_descriptors = np.hstack([all_soap_descriptors, all_forces])
else:
    all_descriptors = all_soap_descriptors

metadata_df = build_provenance_table(structures, vasp_files, fixed_mask_fn=fixed_mask)

# Descriptor matrix shape
print(f"Total descriptors computed: {all_descriptors.shape[0]}")
print(f"Descriptor matrix shape: {all_descriptors.shape}")

# Meta data shape
print(f"Metadata DataFrame shape: {metadata_df.shape}")
print("Metadata DataFrame columns:", metadata_df.columns.tolist())


## Save outputs


In [ ]:
# --- outputs
path_to_results = ensure_directory("desc")
clean_desc_directory = True
if clean_desc_directory:
    removed = clear_directory_outputs(
        path_to_results,
        patterns=(
            "*.npy",
            "*_provenance.parquet",
            "*_provenance.csv",
            "*_filemap.json",
            "*_params.txt",
            "*_config.json",
        ),
    )
    if removed:
        print(f"Removed {len(removed)} previous descriptor output files from {path_to_results}")
    else:
        print(f"No previous descriptor output files found in {path_to_results}")
base = f"SOAP_{'-'.join(new_species)}"
run_artifacts = save_descriptor_run(
    path_to_results,
    all_descriptors,
    metadata_df,
    build_filemap(vasp_files),
    {
        "input_roots": input_roots,
        "soap_params": soap_params,
        "include_forces": include_forces,
        "force_components": list(force_components_idx),
        "soap_dim": int(all_soap_descriptors.shape[1]),
        "force_dim": int(all_descriptors.shape[1] - all_soap_descriptors.shape[1]) if include_forces else 0,
        "total_structures": total_structs,
        "clean_desc_directory": clean_desc_directory,
    },
    base_name=base,
    clear_existing=True,
)

npy_file = run_artifacts["npy"]
parquet_file = run_artifacts["provenance"]
files_json = run_artifacts["filemap"]
txt_file = run_artifacts["log"]
print("Saved outputs:")
print(run_artifacts)
